<a href="https://colab.research.google.com/github/malikshahzaib263/neurofive-ml-track/blob/main/Week_5_Task_2_Streamlit_Model_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 – Machine Learning Fundamentals

# Task 2: Deploy Your Model as a Live Web App

### Neurofive Solutions – Machine Learning Track

**Author:** Shahzaib Arshad

---

## Project Overview

In this project, I deploy a previously trained machine learning model as an interactive web application using Streamlit.

The Titanic Survival Prediction pipeline developed in Week 4 is used for deployment. The application allows users to enter passenger information and receive a survival prediction from the trained machine learning model.

The final application will be deployed using Streamlit Community Cloud and made publicly accessible through a live web link.

## Project Objectives

- Use a previously trained machine learning pipeline.
- Load the saved model using Joblib.
- Build an interactive Streamlit web application.
- Create user-friendly input fields.
- Generate predictions through a Predict button.
- Display survival prediction and probability.
- Prepare deployment files.
- Push the application to GitHub.
- Deploy the application using Streamlit Community Cloud.

In [1]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.4 MB/s eta 0:00:00


In [2]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import sklearn

print("Streamlit version:", st.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Setup completed successfully.")

Streamlit version: 1.61.1
Scikit-learn version: 1.6.1
Setup completed successfully.


In [3]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [4]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset loaded successfully!
Rows: 891
Columns: 12


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
print(df.columns.tolist())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


## Feature Engineering

Two additional features are created for the Titanic survival model:

- **FamilySize:** Total number of family members travelling together, including the passenger.
- **IsAlone:** Indicates whether the passenger was travelling alone.

These engineered features may help the model capture the relationship between family structure and survival.

In [7]:
df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

df[
    [
        "SibSp",
        "Parch",
        "FamilySize",
        "IsAlone"
    ]
].head(10)

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1
5,0,0,1,1
6,0,0,1,1
7,3,1,5,0
8,0,2,3,0
9,1,0,2,0


In [8]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone"
]

X = df[features].copy()

y = df["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 9)
y shape: (891,)


In [9]:
numerical_columns = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_columns = [
    "Sex",
    "Embarked"
]

print("Numerical:", numerical_columns)
print("Categorical:", categorical_columns)

Numerical: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
Categorical: ['Sex', 'Embarked']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 712
Testing samples: 179


In [11]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [12]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_columns
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [14]:
final_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("ML Pipeline created successfully!")

ML Pipeline created successfully!


In [15]:
final_pipeline.fit(
    X_train,
    y_train
)

print("Model trained successfully!")

Model trained successfully!


In [16]:
predictions = final_pipeline.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    f"Model Accuracy: {accuracy * 100:.2f}%"
)

Model Accuracy: 80.45%


In [17]:
print(
    classification_report(
        y_test,
        predictions,
        target_names=[
            "Did Not Survive",
            "Survived"
        ]
    )
)

                 precision    recall  f1-score   support

Did Not Survive       0.82      0.88      0.85       110
       Survived       0.78      0.68      0.73        69

       accuracy                           0.80       179
      macro avg       0.80      0.78      0.79       179
   weighted avg       0.80      0.80      0.80       179



In [18]:
model_filename = "titanic_final_pipeline.joblib"

joblib.dump(
    final_pipeline,
    model_filename
)

print(
    "Model saved successfully as:",
    model_filename
)

Model saved successfully as: titanic_final_pipeline.joblib


In [19]:
print(
    "File exists:",
    os.path.exists(
        "titanic_final_pipeline.joblib"
    )
)

print(
    "File size:",
    os.path.getsize(
        "titanic_final_pipeline.joblib"
    ),
    "bytes"
)

File exists: True
File size: 5098 bytes


In [20]:
loaded_model = joblib.load(
    "titanic_final_pipeline.joblib"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [21]:
test_passenger = pd.DataFrame({
    "Pclass": [1],
    "Sex": ["female"],
    "Age": [28.0],
    "SibSp": [0],
    "Parch": [0],
    "Fare": [80.0],
    "Embarked": ["C"],
    "FamilySize": [1],
    "IsAlone": [1]
})

test_passenger

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,1,female,28.0,0,0,80.0,C,1,1


In [22]:
prediction = loaded_model.predict(
    test_passenger
)[0]

probability = loaded_model.predict_proba(
    test_passenger
)[0][1]

if prediction == 1:
    print("Prediction: SURVIVED")
else:
    print("Prediction: DID NOT SURVIVE")

print(
    f"Survival Probability: "
    f"{probability * 100:.2f}%"
)

Prediction: SURVIVED
Survival Probability: 93.74%


In [23]:
import streamlit as st

print("Streamlit version:", st.__version__)
print("Streamlit installed successfully!")

Streamlit version: 1.61.1
Streamlit installed successfully!


In [24]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# ==========================================
# PAGE CONFIGURATION
# ==========================================

st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

# ==========================================
# LOAD TRAINED MODEL
# ==========================================

@st.cache_resource
def load_model():
    return joblib.load("titanic_final_pipeline.joblib")

model = load_model()

# ==========================================
# HEADER
# ==========================================

st.title("🚢 Titanic Survival Predictor")

st.write(
    """
    This machine learning web application predicts whether
    a Titanic passenger was likely to survive based on
    passenger information.
    """
)

st.info(
    "Enter the passenger details below and click "
    "'Predict Survival' to generate a prediction."
)

st.divider()

# ==========================================
# PASSENGER INFORMATION
# ==========================================

st.subheader("👤 Passenger Information")

pclass = st.selectbox(
    "Passenger Class",
    options=[1, 2, 3],
    format_func=lambda x: {
        1: "1st Class",
        2: "2nd Class",
        3: "3rd Class"
    }[x]
)

sex = st.selectbox(
    "Sex",
    options=["male", "female"]
)

age = st.number_input(
    "Age",
    min_value=0.0,
    max_value=100.0,
    value=30.0,
    step=1.0
)

sibsp = st.number_input(
    "Number of Siblings / Spouses",
    min_value=0,
    max_value=10,
    value=0,
    step=1
)

parch = st.number_input(
    "Number of Parents / Children",
    min_value=0,
    max_value=10,
    value=0,
    step=1
)

fare = st.number_input(
    "Ticket Fare",
    min_value=0.0,
    max_value=600.0,
    value=32.0,
    step=1.0
)

embarked = st.selectbox(
    "Port of Embarkation",
    options=["S", "C", "Q"],
    format_func=lambda x: {
        "S": "Southampton (S)",
        "C": "Cherbourg (C)",
        "Q": "Queenstown (Q)"
    }[x]
)

# ==========================================
# FEATURE ENGINEERING
# ==========================================

family_size = sibsp + parch + 1

is_alone = 1 if family_size == 1 else 0

st.divider()

st.subheader("👨‍👩‍👧 Family Information")

col1, col2 = st.columns(2)

with col1:
    st.metric(
        "Family Size",
        family_size
    )

with col2:
    st.metric(
        "Travelling Alone",
        "Yes" if is_alone == 1 else "No"
    )

# ==========================================
# CREATE MODEL INPUT
# ==========================================

input_data = pd.DataFrame({
    "Pclass": [pclass],
    "Sex": [sex],
    "Age": [age],
    "SibSp": [sibsp],
    "Parch": [parch],
    "Fare": [fare],
    "Embarked": [embarked],
    "FamilySize": [family_size],
    "IsAlone": [is_alone]
})

# ==========================================
# PREDICTION
# ==========================================

st.divider()

if st.button(
    "🔍 Predict Survival",
    type="primary",
    use_container_width=True
):

    prediction = model.predict(
        input_data
    )[0]

    probabilities = model.predict_proba(
        input_data
    )[0]

    survival_probability = (
        probabilities[1] * 100
    )

    non_survival_probability = (
        probabilities[0] * 100
    )

    st.subheader("🎯 Prediction Result")

    if prediction == 1:

        st.success(
            "✅ The passenger is predicted to SURVIVE."
        )

    else:

        st.error(
            "❌ The passenger is predicted NOT TO SURVIVE."
        )

    st.metric(
        "Survival Probability",
        f"{survival_probability:.2f}%"
    )

    st.progress(
        int(survival_probability)
    )

    with st.expander(
        "View Prediction Details"
    ):

        st.write(
            f"Survival Probability: "
            f"{survival_probability:.2f}%"
        )

        st.write(
            f"Non-Survival Probability: "
            f"{non_survival_probability:.2f}%"
        )

# ==========================================
# ABOUT
# ==========================================

st.divider()

with st.expander("ℹ️ About This Project"):

    st.write(
        """
        This application uses a Logistic Regression
        machine learning model trained on the Titanic
        dataset.

        A complete scikit-learn Pipeline is used for
        preprocessing and prediction. Numerical features
        are scaled using StandardScaler, while categorical
        features are encoded using OneHotEncoder.

        Two engineered features are also used:
        FamilySize and IsAlone.
        """
    )

st.caption(
    "Developed for the Neurofive Solutions "
    "Machine Learning Track"
)

Writing app.py


In [25]:
import os

print(
    "app.py exists:",
    os.path.exists("app.py")
)

print(
    "app.py size:",
    os.path.getsize("app.py"),
    "bytes"
)

app.py exists: True
app.py size: 4756 bytes


In [26]:
!python -m py_compile app.py

In [27]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib

Writing requirements.txt


In [28]:
!cat requirements.txt

streamlit
pandas
numpy
scikit-learn
joblib


In [29]:
!ls -lh

total 88K
-rw-r--r-- 1 root root 4.7K Aug 12 20:30 app.py
drwxr-xr-x 2 root root 4.0K Aug 12 20:31 __pycache__
-rw-r--r-- 1 root root   43 Aug 12 20:31 requirements.txt
drwxr-xr-x 1 root root 4.0K Aug 10 13:26 sample_data
-rw-r--r-- 1 root root 5.0K Aug 12 20:29 titanic_final_pipeline.joblib
-rw-r--r-- 1 root root  60K Aug 12 20:25 train.csv


In [30]:
import pandas as pd
import joblib

deployment_model = joblib.load(
    "titanic_final_pipeline.joblib"
)

deployment_test = pd.DataFrame({
    "Pclass": [1],
    "Sex": ["female"],
    "Age": [28.0],
    "SibSp": [0],
    "Parch": [0],
    "Fare": [80.0],
    "Embarked": ["C"],
    "FamilySize": [1],
    "IsAlone": [1]
})

prediction = deployment_model.predict(
    deployment_test
)[0]

probability = deployment_model.predict_proba(
    deployment_test
)[0][1]

print("Deployment Test")
print("-" * 30)
print("Prediction:", prediction)
print(
    f"Survival Probability: "
    f"{probability * 100:.2f}%"
)

Deployment Test
------------------------------
Prediction: 1
Survival Probability: 93.74%


In [31]:
import os
import shutil

folder_name = "Week_5_Task_2_Streamlit_Deployment"

os.makedirs(
    folder_name,
    exist_ok=True
)

print("Folder created:", folder_name)

Folder created: Week_5_Task_2_Streamlit_Deployment


In [32]:
shutil.copy(
    "app.py",
    f"{folder_name}/app.py"
)

shutil.copy(
    "requirements.txt",
    f"{folder_name}/requirements.txt"
)

shutil.copy(
    "titanic_final_pipeline.joblib",
    f"{folder_name}/titanic_final_pipeline.joblib"
)

print("Deployment files copied successfully!")

Deployment files copied successfully!


In [33]:
%%writefile Week_5_Task_2_Streamlit_Deployment/README.md

# Titanic Survival Predictor

This project is an interactive Machine Learning web application developed using Streamlit.

## Model

The application uses a Logistic Regression model trained on the Titanic dataset.

## Features

The model uses:

- Passenger Class
- Sex
- Age
- Siblings / Spouses
- Parents / Children
- Fare
- Port of Embarkation
- Family Size
- Travelling Alone status

## Feature Engineering

Two engineered features are used:

- FamilySize
- IsAlone

## Technologies

- Python
- Pandas
- Scikit-learn
- Streamlit
- Joblib

## Deployment

The application is deployed using Streamlit Community Cloud.

## Author

Shahzaib Arshad

Neurofive Solutions – Machine Learning Track

Writing Week_5_Task_2_Streamlit_Deployment/README.md


In [34]:
!find Week_5_Task_2_Streamlit_Deployment -maxdepth 1 -type f -printf "%f\n"

requirements.txt
titanic_final_pipeline.joblib
README.md
app.py


In [35]:
shutil.make_archive(
    "Week_5_Task_2_Streamlit_Deployment",
    "zip",
    "Week_5_Task_2_Streamlit_Deployment"
)

print("Deployment ZIP created!")

Deployment ZIP created!


In [36]:
print(
    os.path.exists(
        "Week_5_Task_2_Streamlit_Deployment.zip"
    )
)

True


In [37]:
from google.colab import files

files.download(
    "Week_5_Task_2_Streamlit_Deployment.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>